# Fondasi Machine Learning

## Bagian 2c: Validasi Silang

### Tujuan Pembelajaran

- merangkai beberapa langkah pengolahan data dalam satu objek;
- menilai model dengan validasi silang;
- menyetel hyperparameter memakai data validasi.

In [ ]:
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")

### Memuat data

Data harga rumah Boston dipakai sebagai contoh. Berkas diunduh sekali, lalu dibaca sebagai pickle.

In [ ]:
data_path = Path("data/boston_housing_clean.pickle")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/boston_housing_clean.pickle"

if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(data_url, data_path)

bundle = pd.read_pickle(data_path)
housing = bundle["dataframe"]
housing.head()

### Pertanyaan

Bila kita ingin memperkirakan *mean squared error* sebuah model regresi linear, cukupkah satu kali pembagian data latih dan uji? Satu pembagian membuat hasil sangat bergantung pada kebetulan pemilihan baris. Validasi silang mengulang pembagian beberapa kali sehingga estimasi galat lebih stabil.

### Validasi silang manual dengan `KFold`

`KFold` membagi indeks data menjadi beberapa lipatan. Setiap lipatan bergiliran menjadi data uji, sisanya menjadi data latih.

In [ ]:
features = housing.drop(columns="MEDV")
target = housing["MEDV"]

folds = KFold(shuffle=True, random_state=72018, n_splits=3)

for fold_number, (train_index, test_index) in enumerate(folds.split(features), start=1):
    print(f"Lipatan {fold_number}")
    print("  indeks latih:", train_index[:10], "jumlah", len(train_index))
    print("  indeks uji  :", test_index[:10], "jumlah", len(test_index))

In [ ]:
manual_scores = []
model = LinearRegression()

for train_index, test_index in folds.split(features):
    X_train, X_test = features.iloc[train_index], features.iloc[test_index]
    y_train, y_test = target[train_index], target[test_index]

    model.fit(X_train, y_train)
    manual_scores.append(r2_score(y_test.to_numpy(), model.predict(X_test)))

manual_scores

Cara di atas bekerja, tetapi repetitif. Bila data perlu distandarkan, langkahnya bertambah lagi.

### Pertanyaan lanjutan

Bagaimana menambahkan standardisasi pada setiap lipatan? Standardisasi harus di-fit pada data latih lipatan itu saja, lalu diterapkan ke data ujinya. Bila scaler di-fit pada seluruh data sebelum pembagian, informasi data uji ikut bocor ke proses pelatihan.

In [ ]:
scaled_scores = []
model = LinearRegression()
scaler = StandardScaler()

for train_index, test_index in folds.split(features):
    X_train, X_test = features.iloc[train_index], features.iloc[test_index]
    y_train, y_test = target[train_index], target[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    model.fit(X_train_scaled, y_train)
    X_test_scaled = scaler.transform(X_test)

    scaled_scores.append(r2_score(y_test.to_numpy(), model.predict(X_test_scaled)))

scaled_scores

Skornya sama dengan sebelumnya. Untuk regresi linear tanpa regularisasi, penskalaan tidak mengubah performa karena modelnya setara secara matematis. Pada model berpenalti, penskalaan berpengaruh besar.

### `Pipeline` dan `cross_val_predict`

`Pipeline` merangkai langkah yang punya metode `fit`, lalu `cross_val_predict` menjalankan seluruh rangkaian itu pada setiap lipatan dan mengumpulkan prediksi di luar sampel.

In [ ]:
estimator = Pipeline([
    ("scaler", StandardScaler()),
    ("regression", LinearRegression()),
])

predictions = cross_val_predict(estimator, features, target, cv=folds)
print("R2 prediksi silang :", r2_score(target, predictions))
print("Rata-rata manual   :", np.mean(scaled_scores))

Kedua angka hampir identik. Perbedaannya: `cross_val_predict` memakai model berbeda untuk setiap baris, yaitu model yang tidak pernah melihat baris tersebut saat latihan.

## Penyetelan hyperparameter

Penyetelan hyperparameter memakai validasi silang untuk memilih nilai yang memberi performa terbaik di luar sampel. Contohnya parameter `alpha` pada Lasso.

In [ ]:
alphas = np.geomspace(1e-9, 1e0, num=10)
alphas

In [ ]:
lasso_scores = []

for alpha in alphas:
    lasso_estimator = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=alpha, max_iter=100000)),
    ])
    lasso_predictions = cross_val_predict(lasso_estimator, features, target, cv=folds)
    lasso_scores.append(r2_score(target, lasso_predictions))

list(zip(alphas, lasso_scores))

In [ ]:
print("Koefisien alpha 1e-6:", np.round(Lasso(alpha=1e-6).fit(features, target).coef_, 3))
print("Koefisien alpha 1.0 :", np.round(Lasso(alpha=1.0).fit(features, target).coef_, 3))

Alpha kecil membuat Lasso berperilaku seperti regresi linear biasa. Alpha besar menekan koefisien hingga banyak yang menjadi nol, tanda fitur dianggap tidak penting.

In [ ]:
plt.figure(figsize=(8, 5))
plt.semilogx(alphas, lasso_scores, "-o")
plt.xlabel("alpha")
plt.ylabel("R squared")
plt.title("Pengaruh alpha pada Lasso")
plt.show()

### Latihan: tambahkan fitur polinomial

Sisipkan `PolynomialFeatures` ke dalam pipeline, lalu ulangi validasi silang dengan rentang alpha yang sesuai.

In [ ]:
poly_alphas = np.geomspace(0.06, 6.0, 20)
poly_scores = []

for alpha in poly_alphas:
    poly_estimator = Pipeline([
        ("scaler", StandardScaler()),
        ("polynomial", PolynomialFeatures(degree=3)),
        ("lasso", Lasso(alpha=alpha, max_iter=100000)),
    ])
    poly_predictions = cross_val_predict(poly_estimator, features, target, cv=folds)
    poly_scores.append(r2_score(target, poly_predictions))

plt.semilogx(poly_alphas, poly_scores)
plt.xlabel("alpha")
plt.ylabel("R squared")
plt.title("Lasso dengan fitur polinomial derajat 3")
plt.show()

list(zip(poly_alphas, poly_scores))[:5]

Setelah alpha terbaik ditemukan, model dilatih ulang pada seluruh data sebelum dipakai memprediksi data baru.

In [ ]:
best_estimator = Pipeline([
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=2)),
    ("lasso", Lasso(alpha=0.03)),
])

best_estimator.fit(features, target)
print("R2 pada seluruh data:", best_estimator.score(features, target))
print("Jumlah koefisien nol:", int((best_estimator.named_steps["lasso"].coef_ == 0).sum()))

### Latihan: bandingkan dengan Ridge

Ulangi penyetelan alpha memakai `Ridge`. Ridge tidak memaksa koefisien menjadi nol, tetapi menyusutkannya secara halus.

In [ ]:
ridge_alphas = np.geomspace(4, 20, 20)
ridge_scores = []

for alpha in ridge_alphas:
    ridge_estimator = Pipeline([
        ("scaler", StandardScaler()),
        ("polynomial", PolynomialFeatures(degree=2)),
        ("ridge", Ridge(alpha=alpha, max_iter=100000)),
    ])
    ridge_predictions = cross_val_predict(ridge_estimator, features, target, cv=folds)
    ridge_scores.append(r2_score(target, ridge_predictions))

plt.plot(ridge_alphas, ridge_scores)
plt.xlabel("alpha")
plt.ylabel("R squared")
plt.title("Ridge dengan fitur polinomial derajat 2")
plt.show()

best_ridge_alpha = ridge_alphas[int(np.argmax(ridge_scores))]
best_ridge_alpha, max(ridge_scores)

Kesimpulan: Lasso dan Ridge yang alpha-nya disetel dengan benar sama-sama mengalahkan regresi linear biasa, karena keduanya menahan overfitting akibat fitur polinomial.

### Melihat fitur paling berpengaruh

Koefisien Lasso yang tidak nol menunjukkan fitur yang benar-benar dipakai model.

In [ ]:
feature_names = best_estimator.named_steps["polynomial"].get_feature_names_out(features.columns)
importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": best_estimator.named_steps["lasso"].coef_,
})
importance.reindex(importance["coefficient"].abs().sort_values(ascending=False).index).head(10)

## Grid Search CV

Dua pendekatan sebelumnya memakai `KFold` manual dan `cross_val_predict`. Cara paling ringkas adalah `GridSearchCV`, yang menyisir seluruh kombinasi parameter dengan validasi silang.

In [ ]:
grid_estimator = Pipeline([
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures()),
    ("ridge", Ridge()),
])

param_grid = {
    "polynomial__degree": [1, 2, 3],
    "ridge__alpha": np.geomspace(4, 20, 20),
}

grid = GridSearchCV(grid_estimator, param_grid, cv=folds)
grid.fit(features, target)

grid.best_score_, grid.best_params_

In [ ]:
grid_predictions = grid.predict(features)
print("R2 in-sample dan out-of-sample gabungan:", r2_score(target, grid_predictions))
print("Koefisien model terbaik (5 pertama):", np.round(grid.best_estimator_.named_steps["ridge"].coef_[:5], 3))

In [ ]:
grid.cv_results_["mean_test_score"][:5]

## Ringkasan

1. Lipatan validasi bisa dibuat manual dengan `KFold`.
2. `cross_val_predict` merangkum seluruh proses validasi silang dalam satu panggilan.
3. `Pipeline` menjaga setiap langkah pengolahan tetap berada di dalam lipatan, sehingga tidak ada kebocoran data.
4. `GridSearchCV` menyetel hyperparameter sekaligus memilih model terbaik.